# Бейзлайн

Задача: для каждого маршрута `route_id` спрогнозировать `target_1h` на 8 следующих точек
с шагом 30 минут после конца train. Строки теста перечислены в `test_solo_track.parquet`.

Посылка - CSV `sample_submission.csv` с колонками `id,y_pred`: ровно одна строка на каждый `id` из теста.

Метрика - **WAPE + |Relative Bias|**, меньше - лучше:

$$\text{score} = \frac{\sum |\hat y - y|}{\sum y} + \left|\frac{\sum \hat y}{\sum y} - 1\right|$$

Если окружение ещё не настроено:

```python
# %pip install pandas pyarrow numpy matplotlib scikit-learn jupyter ipykernel
```

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
plt.rcParams["figure.figsize"] = (12, 5)

DATA_DIR = Path(".")
TRAIN_PATH = DATA_DIR / "train_solo_track.parquet"
TEST_PATH = DATA_DIR / "test_solo_track.parquet"
SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"

TARGET_COL = "target_1h"
FORECAST_POINTS = 8  # сколько точек по 30 минут прогнозируем для каждого маршрута
STEP = pd.Timedelta(minutes=30)
FUTURE_TARGET_COLS = [f"target_step_{step}" for step in range(1, FORECAST_POINTS + 1)]

TRAIN_DAYS = 14
MAX_TRAIN_ROWS = 1_500_000
RIDGE_ALPHA = 4.0
RANDOM_STATE = 42

## Загрузка данных

In [ ]:
train_df = pd.read_parquet(TRAIN_PATH)
test_df = pd.read_parquet(TEST_PATH)

train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"] = pd.to_datetime(test_df["timestamp"])

train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

print("train shape:", train_df.shape)
print("test shape:", test_df.shape)

In [ ]:
display(train_df.head())
display(test_df.head())

In [ ]:
print("Train date range:", train_df["timestamp"].min(), "->", train_df["timestamp"].max())
print("Test date range:", test_df["timestamp"].min(), "->", test_df["timestamp"].max())
print("Train routes:", train_df["route_id"].nunique())
print("Test routes:", test_df["route_id"].nunique())
print("Test points per route:", test_df.groupby("route_id").size().unique())

## EDA train-данных

In [ ]:
overview = pd.DataFrame({
    "dtype": train_df.dtypes.astype(str),
    "missing_cnt": train_df.isna().sum(),
    "missing_pct": (train_df.isna().mean() * 100).round(4),
    "n_unique": train_df.nunique(dropna=False),
})
overview

In [ ]:
status_cols = sorted(col for col in train_df.columns if col.startswith("status_"))
print("Status columns:", status_cols)
print("Target column:", TARGET_COL)

## Распределения

In [ ]:
fig, axes = plt.subplots(len(status_cols), 1, figsize=(14, 3 * len(status_cols)))

for ax, col in zip(np.atleast_1d(axes), status_cols, strict=True):
    ax.hist(train_df[col].clip(upper=train_df[col].quantile(0.99)), bins=30)
    ax.set_title(f"{col} (clipped at p99)")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 4))
plt.hist(train_df[TARGET_COL].clip(upper=train_df[TARGET_COL].quantile(0.99)), bins=25)
plt.title(f"{TARGET_COL} (clipped at p99)")
plt.show()

## Будущие таргеты

Для каждой строки train добавляем значения таргета через 1, 2, ... шагов вперёд по тому же маршруту.
Модель по текущему состоянию маршрута сразу предсказывает все шаги горизонта.

In [ ]:
route_group = train_df.groupby("route_id", sort=False)

for step in range(1, FORECAST_POINTS + 1):
    train_df[f"target_step_{step}"] = route_group[TARGET_COL].shift(-step)

train_df[["route_id", "timestamp", TARGET_COL] + FUTURE_TARGET_COLS].head(10)

In [ ]:
supervised_df = train_df.dropna(subset=FUTURE_TARGET_COLS).copy()
print("Rows with future targets:", supervised_df.shape)

## Корреляции

In [ ]:
corr_cols = status_cols + FUTURE_TARGET_COLS
corr = supervised_df[corr_cols].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_cols)), corr_cols, rotation=90)
ax.set_yticks(range(len(corr_cols)), corr_cols)
for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7)
fig.colorbar(im, ax=ax)
ax.set_title("Correlations on supervised train")
plt.show()

## Подготовка train и test

In [ ]:
feature_cols = [col for col in train_df.columns if col not in {TARGET_COL, "timestamp", "id", *FUTURE_TARGET_COLS}]
print("Feature columns:", feature_cols)

In [ ]:
train_model_df = supervised_df[feature_cols + ["timestamp"] + FUTURE_TARGET_COLS].copy()
train_model_df = train_model_df.rename(columns={"timestamp": "source_timestamp"})

train_ts_max = train_model_df["source_timestamp"].max()
train_window_start = train_ts_max - pd.Timedelta(days=TRAIN_DAYS)
train_model_df = train_model_df[train_model_df["source_timestamp"] >= train_window_start].copy()

print("Recent train rows:", train_model_df.shape)

In [ ]:
# прогноз строим от последнего известного состояния каждого маршрута
test_model_df = train_df.groupby("route_id", sort=False).tail(1).copy()
assert test_model_df["route_id"].nunique() == test_df["route_id"].nunique()

print("Inference rows:", test_model_df.shape)

## Time-based split

In [ ]:
train_model_df = train_model_df.sort_values("source_timestamp").copy()
split_point = train_model_df["source_timestamp"].quantile(0.8)

fit_df = train_model_df[train_model_df["source_timestamp"] <= split_point].copy()
valid_df = train_model_df[train_model_df["source_timestamp"] > split_point].copy()

if len(fit_df) > MAX_TRAIN_ROWS:
    fit_df = fit_df.sample(MAX_TRAIN_ROWS, random_state=RANDOM_STATE)

print("Fit rows:", fit_df.shape)
print("Valid rows:", valid_df.shape)

In [ ]:
X_fit = fit_df[feature_cols]
y_fit = fit_df[FUTURE_TARGET_COLS]

X_valid = valid_df[feature_cols]
y_valid = valid_df[FUTURE_TARGET_COLS]

X_test = test_model_df[feature_cols]

## Линейный baseline

In [ ]:
categorical_features = [col for col in feature_cols if col.endswith("_id")]
numeric_features = [col for col in feature_cols if col not in categorical_features]

print("Categorical features:", categorical_features)
print("Numeric features:", numeric_features)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]),
            numeric_features,
        ),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

model = Pipeline(steps=[("preprocessor", preprocessor), ("regressor", Ridge(alpha=RIDGE_ALPHA))])

In [ ]:
model.fit(X_fit, y_fit)

In [ ]:
def predict(X: pd.DataFrame, index: pd.Index) -> pd.DataFrame:
    """Прогноз на все шаги горизонта; таргет неотрицательный, поэтому обрезаем снизу нулём."""
    return pd.DataFrame(model.predict(X), columns=FUTURE_TARGET_COLS, index=index).clip(lower=0)


fit_pred_df = predict(X_fit, fit_df.index)
valid_pred_df = predict(X_valid, valid_df.index)
test_pred_df = predict(X_test, test_model_df.index)

valid_pred_df.head()

## Метрика

Та же формула, что на платформе: WAPE + |Relative Bias|, меньше - лучше.
На платформе она считается по всем точкам посылки сразу (отдельно на Public и Private частях теста).

In [ ]:
def wape_rbias(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """WAPE + |Relative Bias|, меньше - лучше."""
    y_true = np.asarray(y_true, dtype=float).ravel()
    y_pred = np.asarray(y_pred, dtype=float).ravel()
    wape = np.abs(y_pred - y_true).sum() / y_true.sum()
    rbias = np.abs(y_pred.sum() / y_true.sum() - 1)
    return wape + rbias


def report(name: str, y_true: pd.DataFrame, y_pred: pd.DataFrame) -> None:
    """Печатает общую метрику и метрику по каждому шагу горизонта."""
    by_step = pd.Series({col: wape_rbias(y_true[col], y_pred[col]) for col in FUTURE_TARGET_COLS})
    print(f"{name}: общая метрика = {wape_rbias(y_true, y_pred):.4f}")
    print("по горизонтам:")
    display(by_step.round(4).to_frame("wape_rbias").T)


report("Train (fit)", y_fit, fit_pred_df)
report("Валидация", y_valid, valid_pred_df)

## Конвертируем прогноз в формат посылки

In [ ]:
test_pred_df["route_id"] = test_model_df["route_id"]
test_pred_df["last_timestamp"] = test_model_df["timestamp"]

# разворачиваем target_step_* в строки: шаг k -> время последнего факта + k * 30 минут
forecast_df = test_pred_df.melt(
    id_vars=["route_id", "last_timestamp"],
    value_vars=FUTURE_TARGET_COLS,
    var_name="step",
    value_name="y_pred",
)
forecast_df["step_num"] = forecast_df["step"].str.removeprefix("target_step_").astype(int)
forecast_df["timestamp"] = forecast_df["last_timestamp"] + forecast_df["step_num"] * STEP

submission = (
    test_df[["id", "route_id", "timestamp"]]
    .merge(forecast_df[["route_id", "timestamp", "y_pred"]], on=["route_id", "timestamp"], how="left")
    .sort_values("id")[["id", "y_pred"]]
    .reset_index(drop=True)
)
submission.head()

In [ ]:
# проверяем, что посылка в формате платформы: по одной строке на каждый id теста, без пропусков
assert len(submission) == len(test_df)
assert submission["id"].is_unique
assert set(submission["id"]) == set(test_df["id"])
assert submission["y_pred"].notna().all(), "для части точек теста не нашлось прогноза"

## Выгрузка CSV

In [ ]:
submission.to_csv(SUBMISSION_PATH, index=False)
print("submission saved to:", SUBMISSION_PATH)

Файл `sample_submission.csv` (колонки `id,y_pred`) можно отправлять на платформу.